In [12]:
import pandas as pd
import os
import re

In [13]:

csv_files = [f for f in os.listdir("../outputs/") if f.endswith(".csv")]
csv_files

['temp_results_SC_chunk2of10_4869 2.csv',
 'temp_results_HI_35494 2.csv',
 'temp_results_RI_chunk1of4_3428.csv',
 'temp_results_TN_chunk8of9_3384.csv',
 'state_results_CT_chunk4of9.csv',
 'temp_results_AR_chunk8_72657.csv',
 'state_results_AZ_chunk14.csv',
 'temp_results_WA_chunk51of51_3384.csv',
 'temp_results_WA_chunk8of55_4893.csv',
 'state_results_ME_chunk6of9.csv',
 'temp_results_ME_chunk6of9_3400.csv',
 'state_results_WV_chunk3of7.csv',
 'state_results_ND_chunk5of8.csv',
 'temp_results_MA_65808.csv',
 'temp_results_AZ_chunk16_72653.csv',
 'state_results_MI 2.csv',
 'state_results_IL 2.csv',
 'temp_results_AL_chunk3_51835.csv',
 'state_results_PA_chunk1of8.csv',
 'temp_results_CO_12489.csv',
 'temp_results_AL_chunk14_51834.csv',
 'temp_results_TN_chunk4of9_3382.csv',
 'temp_results_WA_chunk16of51_3434.csv',
 'state_results_IL_chunk1of3.csv',
 'temp_results_AL_chunk9_72655.csv',
 'state_results_WA_chunk3of51.csv',
 'state_results_AR 2.csv',
 'state_results_MT_chunk3of6.csv',
 'temp

In [14]:
full_df = pd.DataFrame()
for file in csv_files:
    df = pd.read_csv(f"../outputs/{file}")

    full_df = pd.concat([full_df,df])

/var/folders/qd/53v0q6g93r5g_dc_bwl8_tlw0000gn/T/ipykernel_51447/1822516544.py:3: DtypeWarning: Columns (13,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"../outputs/{file}")
/var/folders/qd/53v0q6g93r5g_dc_bwl8_tlw0000gn/T/ipykernel_51447/1822516544.py:3: DtypeWarning: Columns (13,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"../outputs/{file}")


In [15]:
full_df.sort_values(by="SERFF Tracking Number",inplace=True)

In [16]:
df = full_df.copy()

In [17]:
full_df.shape

(1787258, 19)

In [18]:
cols = [
    "form_attachments",
    "supporting_document_attachments",
    "rate_rule_attachments",
    "correspondence_attachments",
    "toc_form_names",
    "readability_text",
]

df = df.dropna(subset=cols, how="all")


In [19]:
df.shape

(1086072, 19)

In [20]:
df.drop_duplicates(inplace=True)

In [21]:
df.shape

(484738, 19)

In [22]:
df1 = df.copy()

In [30]:
df.to_csv("../data/scraped_result_before_explode.csv",index=False)

In [23]:
df.sample(2)

,Company Name,NAIC Company Code,Insurance Product Name,Sub Type Of Insurance,Filing Type,Filing Status,SERFF Tracking Number,Page Number,serf_num,state,page_url,form,auth_url,form_attachments,supporting_document_attachments,rate_rule_attachments,correspondence_attachments,toc_form_names,readability_text
954,The Chesapeake Life Insurance Company,61832.0,Advertising Filing - CH 26118,H14I.000 Health - Hospital Indemnity,Form,Closed - Filed,MGCC-128819395,799,128819395,WA,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/WA,{'Advertising CH IND WA 1212.pdf': 40},NaN,NaN,NaN,"{'Usage Agreement.pdf': 'Usage Agreement', 'Ad...",NaN
408,GPM Health and Life Insurance Company,67059.0,"State of Emergency November 19, 2022-Alpine-In...",H21.000 Health - Other,Form,Closed - Acknowledged for Filing,MUTM-133477019,49,133477019,CA,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/CA,NaN,{'CA Emergency Notice Response -Alpine-Inyo St...,NaN,NaN,"{'Usage Agreement.pdf': 'Usage Agreement', '11...",NaN


In [24]:
import ast
import pandas as pd
import numpy as np

def parse_to_dict(val):
    if isinstance(val, dict):
        return val

    if isinstance(val, str):
        val = val.strip()
        if val.startswith("{") and val.endswith("}"):
            try:
                parsed = ast.literal_eval(val)
                if isinstance(parsed, dict):
                    return parsed
            except Exception:
                pass

    return None


In [25]:
import pandas as pd
import numpy as np

attachment_cols = [
    "form_attachments",
    "supporting_document_attachments",
    "rate_rule_attachments",
    "correspondence_attachments"
]


def merge_attachments(row):
    records = []

    for col in attachment_cols:
        parsed = parse_to_dict(row[col])

        if parsed:
            for filename, pages in parsed.items():
                records.append({
                    "attachment_type": col,
                    "file_name": filename,
                    "number_of_pages": pages
                })

    return records if records else np.nan


df["attachments"] = df.apply(merge_attachments, axis=1)


In [26]:
df['readability_text'].value_counts()

readability_text
{'Readability Certification.pdf': ''}                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [27]:
df.drop(columns=['readability_text'],inplace=True)

In [32]:
df_attach = df[["SERFF Tracking Number","attachments"]]

## Readability

In [33]:
df = df1[['SERFF Tracking Number','toc_form_names', 'readability_text',"state","page_url","auth_url"]]

In [34]:
df.dropna(inplace=True)

/var/folders/qd/53v0q6g93r5g_dc_bwl8_tlw0000gn/T/ipykernel_51447/1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [35]:
df.shape

(141091, 6)

In [36]:
import ast
import pandas as pd
import numpy as np

def all_values_empty(val):
    if pd.isna(val):
        return False  # don't drop NaN rows unless you explicitly want to

    # Convert stringified dict → dict
    if isinstance(val, str):
        try:
            val = ast.literal_eval(val)
        except Exception:
            return False  # not a dict-like string → keep row

    if isinstance(val, dict):
        values = val.values()
        return all(
            v is None or
            (isinstance(v, float) and np.isnan(v)) or
            (isinstance(v, str) and v.strip() == "")
            for v in values
        )

    return False


In [37]:
df = df[~df["readability_text"].apply(all_values_empty)]


In [38]:
import ast
import pandas as pd

def str_dict_to_dict(val):
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass
    return val

df["readability_text"] = df["readability_text"].apply(str_dict_to_dict)


In [39]:
df.values[100]

array(['AAAL-126597886',
       "{'Usage Agreement.pdf': 'Usage Agreement', 'RIDER.pdf': 'ML-10REB RECUPERATION EXTENSION BENEFIT', 'ML-10ABI AIRLINE BENEFIT INCREASE RIDER.pdf': 'Airline Benefit Increase Rider', 'ML-10SBI SPOUSE BENEFIT INCREASE RIDER.pdf': 'Spouse Benefit Increase Rider', 'CertificationofCompliance.pdf': 'Certification of Compliance', 'Act Memo - Additional Accidental Death Benefit Rider.pdf': 'Actuarial Memorandum', 'Act Memo - Additional Accidental Hospital Rider.pdf': 'Actuarial Memorandum', 'Act Memo - Airline Beneift Increase Rider.pdf': 'Actuarial Memorandum', 'Act Memo - Recuperation Extension Benefit Rider.pdf': 'Actuarial Memorandum', 'Act Memo - Spouse Beneift Increase Rider.pdf': 'Actuarial Memorandum', 'Act Memo MLTA  2010.pdf': 'Actuarial Memorandum', 'Readability Certification.pdf': 'Readability Certification', 'Statement of Variability.pdf': 'Statement of Variability', 'WI Cover Letter MLT08  Riders.pdf': 'Cover Letter', 'ML-10SBA Certificate Schedule 

In [75]:
base_prompt = """You are an excellent document expert. You will be provided with an filename to pdf name. For each of the file/pdf there could be a flesch score score given in a unstructured extracted text. 
                    \n
                    Your task is to \n
                    
                    Identify/associate the flesch score with correct file. MAKE SURE YOU MAP TO ONLY THE FILE FOR WHICH THE FLESCH SCORE IS GIVEN. IF SCORE IS NOT AVAILABLE, RETURN NULL\n"""
                    
end_prompt = """\n\n
                  Return the result as json:
                  \n\n{ file_name : <flesch score>, ...}"""

In [76]:
df.shape

(117773, 6)

In [77]:
df.columns

Index(['SERFF Tracking Number', 'toc_form_names', 'readability_text', 'state',
       'page_url', 'auth_url'],
      dtype='object')

In [40]:
df.drop_duplicates(subset=['SERFF Tracking Number'],inplace=True)

In [41]:
import json
import os
import glob
import math
import pandas as pd


In [53]:
import os
import json
import glob

DATA_DIR = "batch_output/"
jsonl_files = glob.glob(os.path.join(DATA_DIR, "*.jsonl"))

results = {}   # serff_id -> {file: score}


In [54]:
def safe_merge(existing: dict, new: dict) -> dict:
    """
    Merge file->score mappings.
    New values overwrite only if existing is None.
    """
    for k, v in new.items():
        if k not in existing or existing[k] is None:
            existing[k] = v
    return existing


In [55]:
for filepath in jsonl_files:
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            serff_id = obj.get("custom_id")
            if not serff_id:
                continue

            try:
                raw = obj["response"]["body"]["output"][1]["content"][0]["text"]
                parsed = json.loads(raw)
            except Exception:
                continue

            if not isinstance(parsed, dict):
                continue

            if serff_id not in results:
                results[serff_id] = parsed
            else:
                results[serff_id] = safe_merge(results[serff_id], parsed)


In [56]:
import pandas as pd

rows = []

for serff, files in results.items():
    for file_name, score in files.items():
        rows.append({
            "serff_num": serff,
            "file_name": file_name,
            "readability_score": score
        })

readability_df = pd.DataFrame(rows)


In [57]:
import pandas as pd

rows = []

for _, row in df_attach.iterrows():
    serff = row["SERFF Tracking Number"]
    attachments = row["attachments"]

    if not isinstance(attachments, list):
        continue

    for att in attachments:
        rows.append({
            "serff_num": serff,
            "file_name": att.get("file_name"),
            "attachment_type": att.get("attachment_type"),
            "number_of_pages": att.get("number_of_pages"),
        })

attachments_long_df = pd.DataFrame(rows)


In [58]:
final_df = attachments_long_df.merge(
    readability_df,
    on=["serff_num", "file_name"],
    how="left"
)


In [59]:
final_df[final_df.readability_score.notna()].tail(5)

,serff_num,file_name,attachment_type,number_of_pages,readability_score
4668766,ZUUG-132476346,ZALIC-MQD-3 Musculoskeletal Questionnaire 5-3-...,form_attachments,2,50
4668767,ZUUG-132476346,ZALIC-MQD-6 Seizure Disorder Questionnaire 5-3...,form_attachments,2,50
4668768,ZUUG-132476346,ZALIC-MQD-5 Skin Cancer Quest (5-3-20).pdf,form_attachments,2,50
4668769,ZUUG-132476346,ZALIC-MQD-4 Cardiac Arrhythmia Questionnaire (...,form_attachments,2,50
4668770,ZUUG-132476346,ZALIC-MQD-2 Diabetes Questionnaire 5.3.20.pdf,form_attachments,2,50


In [60]:
final_df.shape

(4668792, 5)

In [62]:
df1.columns

Index(['Company Name', 'NAIC Company Code', 'Insurance Product Name',
       'Sub Type Of Insurance', 'Filing Type', 'Filing Status',
       'SERFF Tracking Number', 'Page Number', 'serf_num', 'state', 'page_url',
       'form', 'auth_url', 'form_attachments',
       'supporting_document_attachments', 'rate_rule_attachments',
       'correspondence_attachments', 'toc_form_names', 'readability_text'],
      dtype='object')

In [63]:
df1.to_csv("file_level_data_with_flesch.csv",index=False)

In [75]:
cols_needed = [
    "Company Name",
    "NAIC Company Code",
    "Insurance Product Name",
    "Sub Type Of Insurance",
    "Filing Type",
    "Filing Status",
    "SERFF Tracking Number",
    "Page Number",
    "serf_num",
    "state",
    "page_url",
    "form",
    "auth_url",
    "toc_form_names"
]

df1_small = df1[cols_needed].copy()


In [110]:
df1_small["SERFF Tracking Number"] = df1_small["SERFF Tracking Number"].astype(str)
final_df["serff_num"] = final_df["serff_num"].astype(str)


In [111]:
merged_df = df1_small.merge(
    final_df,
    left_on="SERFF Tracking Number",
    right_on="serff_num",
    how="left"
)


In [112]:
merged_df.drop(columns=["serff_num"], inplace=True)


In [116]:
df = merged_df.sample(1000)

In [114]:
merged_df.to_csv("full_data_file_wise.csv",index=False)

In [117]:
df

,Company Name,NAIC Company Code,Insurance Product Name,Sub Type Of Insurance,Filing Type,Filing Status,SERFF Tracking Number,Page Number,serf_num,state,page_url,form,auth_url,toc_form_names,file_name,attachment_type,number_of_pages,readability_score
2945290,Benchmark Insurance Company,41394.0,Benchmark ESL 2019,H12.004 Self-Funded Health Plan,Form,Closed - Approved,MCHU-132215354,13,132215354,NV,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/NV,"{'Usage Agreement.pdf': 'Usage Agreement', 'BI...",BIC R103-2019 rev 04.22.19.pdf,form_attachments,1,NaN
3747579,Principal Life Insurance Company,61271.0,Small Group Dental Outside Market Dental,H10G.000 Health - Dental,Form,Closed - Filed,PRLF-132424431,650,132424431,WA,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/WA,"{'Usage Agreement.pdf': 'Usage Agreement', 'GC...",redlineGC 7302 WA.pdf,supporting_document_attachments,4,NaN
415371,American Public Life Insurance Company,60801.0,Cancer Indemnity,Specified Disease Indemnity,Form,Closed - Disapproved,AFDL-125228970,13,125228970,CT,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/CT,"{'Usage Agreement.pdf': 'Usage Agreement', 'CP...",CT 071007.pdf,supporting_document_attachments,2,NaN
3697143,Priority Health,95561.0,2017 PH HMO + POS_MyPriority Annual Filing,HOrg02I.005D Individual - HMO,Form/Rate,Closed - Approved,PRHL-130538897,184,130538897,MI,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/MI,"{'Usage Agreement.pdf': 'Usage Agreement', 'ra...",Schedule_Indv_2017-5289_HMO~RxPlusSpectrumHeal...,form_attachments,10,NaN
1893736,"Dominion Dental Services, Inc.",95657.0,Oregon PY 2024 HCR Form Filing,H10I.000 Health - Dental,Form,Closed - Approved,DMND-133567765,34,133567765,OR,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/OR,"{'Usage Agreement.pdf': 'Usage Agreement', 'Br...",EOV - Individual Pediatric PPO Coverage Schedu...,form_attachments,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986337,American United Life Insurance Company,60895.0,Statement of Insurability,H11G.004 Other,Form,Closed - Rejected,AULD-133412721,16,133412721,UT,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/UT,"{'Usage Agreement.pdf': 'Usage Agreement', 'G-...",G-35897_Prescription.pdf,form_attachments,1,NaN
191562,Aetna Health of Utah Inc.,95407.0,2014 SG- App/Enroll (HMO),HOrg02G.004F Small Group Only - HMO,Form,Closed - FILED FOR USE,AENX-G129795757,2,G129795757,UT,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/UT,{'Usage Agreement.pdf': 'Usage Agreement'},HI UT EGR68901-32 102714 V001.PDF,supporting_document_attachments,1,NaN
1560210,"Coventry Health Care of Iowa, Inc.",95241.0,Health Care Reform - PPACA Amendment (Individu...,HOrg02I.005B Individual - Point-of-Service (POS),Form,Closed - Approved,CHCI-126798862,26,126798862,IA,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/IA,"{'Usage Agreement.pdf': 'Usage Agreement', 'CH...",CERT OF COMPL CHCIA.pdf,supporting_document_attachments,1,NaN
2920216,Massachusetts Mutual Life Insurance Company,65935.0,EPS/ME-2012,H11I.009 Combined Short Term and Long Term - R...,Form/Rate,Closed - Approved,MASS-128211735,65,128211735,TN,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/TN,"{'Usage Agreement.pdf': 'Usage Agreement', 'EP...",Sample Policy Specifications.pdf,supporting_document_attachments,10,None


In [79]:
import ast
import pandas as pd

def parse_dict_safe(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return {}
    return {}


In [80]:
merged_df["_form_map"] = merged_df["toc_form_names"].apply(parse_dict_safe)


In [81]:
merged_df["form_name"] = merged_df.apply(
    lambda r: r["_form_map"].get(r["file_name"]),
    axis=1
)


In [85]:
merged_df.columns

Index(['Company Name', 'NAIC Company Code', 'Insurance Product Name',
       'Sub Type Of Insurance', 'Filing Type', 'Filing Status',
       'SERFF Tracking Number', 'Page Number', 'serf_num', 'state', 'page_url',
       'form', 'auth_url', 'toc_form_names', 'file_name', 'attachment_type',
       'number_of_pages', 'readability_score', '_form_map', 'form_name'],
      dtype='object')

In [83]:
merged_df.sample().values[0]

array(['American United Life Insurance Company', 60895.0,
       'Group Hospital Indemnity', 'H14G.000 Health - Hospital Indemnity',
       'Form/Rate', 'Closed - (APP)Form Approval', 'AULD-134526805', 28,
       134526805, 'NC',
       'https://filingaccess.serff.com/sfa/search/filingSummary.xhtml?filingId=134526805',
       True, 'https://filingaccess.serff.com/sfa/home/NC',
       "{'Usage Agreement.pdf': 'Usage Agreement', 'HI G 4100 (NC) Policy 07.08.2025.pdf': 'HI G 4100 (NC)', 'HI GC 4100 (NC) Certificate 07.08.2025.pdf': 'HI GC 4100 (NC)', 'HI GC 4100 SCH (NC) Schedule 08.26.2025.pdf': 'HI GC 4100 SCH (NC)', 'OA.Group HI.Rate Manual.20250428.pdf': 'Rate Manual', 'Authorization letter signed.pdf': 'Third Party Authorization', 'OA.Group HI.Act Memo.20250326.pdf': 'Actuarial Memorandum', 'Hospital Indemnity Checklist.pdf': 'Hospital Indemnity Checklist', 'Readability Certification HI_signed.pdf': 'Readability Certification', 'HI Statement of Variability_NC.pdf': 'Statement of Vari

In [87]:
merged_df.shape

(4883715, 20)

In [107]:
merged_df.sample(5)

,Company Name,NAIC Company Code,Insurance Product Name,Sub Type Of Insurance,Filing Type,Filing Status,SERFF Tracking Number,Page Number,serf_num,state,page_url,form,auth_url,file_name,attachment_type,number_of_pages,readability_score,form_name
3309327,"Delta Dental Plan of New Hampshire, Inc.",47079.0,2020 NH Small Group Forms & Rate Filing,H10G.000 Health - Dental,Form/Rate,Closed - Approved,NEDD-132444930,20,132444930,NH,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/NH,ECF - D 010120 BW.pdf,form_attachments,3,NaN,ECF D 010120
4218189,Symetra Life Insurance Company,68608.0,Select Benefits,H21.000 Health - Other,Form,Closed - Approved,SYMT-125774251,94,125774251,TN,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/TN,variability.pdf,supporting_document_attachments,1,NaN,Description of Variables
3243410,Markel Insurance Company,38970.0,Supplemental GAP Product,H24G.001 Any Size Group,Form,Closed - Disapproved-Final,MRKC-132714682,47,132714682,VT,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/VT,SOV MAGP 1000 02 21 Group GAP Application - A&...,supporting_document_attachments,4,NaN,None
3370090,Noridian Mutual Insurance Company,55891.0,Benefit Plan Agreement Metallic SHOP,H21.000 Health - Other,Form,Closed - Informational,NMIN-129968507,63,129968507,ND,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/ND,29317505_MetallicSHOP_NonmasterBPA_HealthOnly_...,form_attachments,12,NaN,None
300783,Aetna Life Insurance Company,60054.0,Stop Loss: MI SG Forms,H12.004 Self-Funded Health Plan,Form/Rate,Closed - Approved,AETN-131030905,9,131030905,MI,https://filingaccess.serff.com/sfa/search/fili...,True,https://filingaccess.serff.com/sfa/home/MI,Attach G (AggSpec) 2017 MI SG.pdf,rate_rule_attachments,4,NaN,2017 SG Rate Manual


In [88]:
merged_df.drop(columns=["toc_form_names","_form_map"], inplace=True)


In [89]:
merged_df.to_csv("final_data_with_form_name.csv",index=False)

In [98]:
merged_df[merged_df.state=="NC"].sample(2).values[0]

array(['UnitedHealthcare Insurance Company', 79413.0,
       'NC PY26 Surest Form Filing', 'H16G.002A Large Group Only - PPO',
       'Form', 'Assigned', 'UHLC-134628190', 217, 134628190, 'NC',
       'https://filingaccess.serff.com/sfa/search/filingSummary.xhtml?filingId=134628190',
       True, 'https://filingaccess.serff.com/sfa/home/NC',
       'RID26.RX.NET.I.BINDBASIC.2021.LG.NC.pdf', 'form_attachments', 16,
       51.1, 'ASIC.2021.LG.NC'], dtype=object)

In [109]:
df1.values[0]

array(['AAA Life Insurance Company', 71854.0, 'PAI',
       'H03G.000 Health - Accidental Death & Dismemberment', 'Form',
       'Closed - Approved', 'AAAL-125053950', 1, 125053950, 'RI',
       'https://filingaccess.serff.com/sfa/search/filingSummary.xhtml?filingId=125053950',
       True, 'https://filingaccess.serff.com/sfa/home/RI', nan,
       "{'Trust Agreement.pdf': 5, 'Group Ins[1].Ck.list1.pdf': 2, 'Statement of Rates.pdf': 1, 'Actuarial Certification.pdf': 1, 'Amended and Restated Trust for the AAA Group Insurance Trust.pdf': 4, 'Submission Letter.pdf': 2}",
       nan, nan,
       "{'Usage Agreement.pdf': 'Usage Agreement', 'Actuarial Certification.pdf': 'Actuarial Certification - Life & A&H', 'Statement of Rates.pdf': 'Premium Rate Sheets - Life & A&H', 'Group Ins[1].Ck.list1.pdf': 'Health Insurance Checklist', 'Submission Letter.pdf': 'Cover Letter', 'Trust Agreement.pdf': 'Trust Documents', 'Trust.pdf': 'Amended and Restated Trust for the AAA Group Insurance'}",
       nan